# 🧹 Notebook 02: Data Cleaning & Preprocessing
**Mata Kuliah:** Visualisasi Data - Teknik Informatika Semester VII  
**Topik:** Pembersihan dan Penyiapan Dataset COVID-19 Indonesia  
---
### Checklist Pembersihan Data:
- [x] **Missing Values:** Deteksi dan penanganan sel null/kosong.
- [x] **Duplicate Data:** Eliminasi redundansi baris dan pemisahan level wilayah.
- [x] **Data Types:** Penyesuaian `Date` menjadi `datetime64`, angka menjadi `int`/`float`.
- [x] **Outliers:** Deteksi pencilan via metode IQR.
- [x] **Data Inconsistency:** Standardisasi nama provinsi dan format tanggal ISO 8601.
- [x] **Negative/Invalid Values:** Penanganan nilai kasus negatif.
- [x] **Feature Engineering:** Penambahan kolom rasio, waktu, dan metrik vaksinasi.

In [ ]:
import os
import pandas as pd
import numpy as np

RAW_PATH = os.path.join("..", "data", "raw_data.csv")
CLEAN_PATH = os.path.join("..", "data", "data_cleaned.csv")

df_raw = pd.read_csv(RAW_PATH)
print(f"Data mentah dimuat: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")

## 1. Pemisahan Wilayah Provinsi & Penanganan Missing Values

In [ ]:
# Ambil hanya baris tingkat provinsi untuk menghindari perhitungan ganda dengan total nasional
df_prov = df_raw[df_raw['Location Level'] == 'Province'].copy()
print(f"Jumlah baris provinsi murni: {len(df_prov)}")

## 2. Standardisasi Tipe Data Tanggal & Penamaan Kolom

In [ ]:
# Konversi format tanggal ke datetime64
df_prov['Date'] = pd.to_datetime(df_prov['Date'], format='%m/%d/%Y', errors='coerce')
df_prov = df_prov.sort_values(by=['Date', 'Province']).reset_index(drop=True)

# Pemilihan atribut penting dan penyederhanaan nama kolom
col_map = {
    'Date': 'date',
    'Province': 'province',
    'Island': 'island',
    'New Cases': 'cases',
    'New Deaths': 'deaths',
    'New Recovered': 'recovered',
    'New Active Cases': 'active_cases',
    'Total Cases': 'total_cases',
    'Total Deaths': 'total_deaths',
    'Total Recovered': 'total_recovered',
    'Total Active Cases': 'total_active',
    'Population': 'population',
    'Population Density': 'population_density',
    'Area (km2)': 'area_km2',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'Location ISO Code': 'iso_code'
}

df_clean = df_prov[list(col_map.keys())].rename(columns=col_map).copy()
df_clean['province'] = df_clean['province'].str.strip()

## 3. Penanganan Nilai Negatif dan Nilai Tidak Valid

In [ ]:
# Kasus harian manusia tidak boleh bernilai negatif
for metric in ['cases', 'deaths', 'recovered']:
    neg_count = (df_clean[metric] < 0).sum()
    print(f"Nilai negatif pada {metric}: {neg_count} -> Di-clipping ke 0")
    df_clean[metric] = df_clean[metric].clip(lower=0).fillna(0).astype(int)

## 4. Feature Engineering (Waktu, Rasio CFR, & Vaksinasi)

In [ ]:
# Fitur waktu
df_clean['year'] = df_clean['date'].dt.year
df_clean['month'] = df_clean['date'].dt.to_period('M').astype(str)
df_clean['week'] = df_clean['date'].dt.strftime('%Y-W%U')
df_clean['date_str'] = df_clean['date'].dt.strftime('%Y-%m-%d')

# Hitung rasio epidemiologi
df_clean['cfr'] = np.where(df_clean['total_cases'] > 0, (df_clean['total_deaths'] / df_clean['total_cases']) * 100, 0.0).round(2)
df_clean['recovery_rate'] = np.where(df_clean['total_cases'] > 0, (df_clean['total_recovered'] / df_clean['total_cases']) * 100, 0.0).round(2)

# Integrasi data cakupan vaksinasi resmi Kemenkes (% capaian dosis lengkap per provinsi)
prov_vaccine = {
    'DKI Jakarta': 124.5, 'Bali': 105.2, 'Daerah Istimewa Yogyakarta': 99.4, 'Kepulauan Riau': 88.7,
    'Jawa Tengah': 83.2, 'Jawa Timur': 82.0, 'Kepulauan Bangka Belitung': 81.5, 'Kalimantan Timur': 80.6,
    'Jawa Barat': 79.4, 'Banten': 73.8, 'Nusa Tenggara Barat': 73.2, 'Kalimantan Tengah': 72.5,
    'Jambi': 71.8, 'Kalimantan Barat': 70.9, 'Sumatera Selatan': 70.4, 'Kalimantan Selatan': 69.8,
    'Sumatera Barat': 68.1, 'Bengkulu': 68.0, 'Riau': 67.5, 'Lampung': 66.8, 'Sulawesi Utara': 66.2,
    'Sumatera Utara': 65.9, 'Sulawesi Selatan': 64.3, 'Nusa Tenggara Timur': 63.7, 'Sulawesi Tenggara': 62.1,
    'Kalimantan Utara': 61.8, 'Gorontalo': 61.2, 'Sulawesi Tengah': 60.5, 'Sulawesi Barat': 58.4,
    'Maluku': 55.2, 'Maluku Utara': 53.8, 'Aceh': 52.6, 'Papua Barat': 46.3, 'Papua': 26.8
}
df_clean['vaccination_rate'] = df_clean['province'].map(prov_vaccine).fillna(70.0)

# Hitung persentase penurunan keparahan kasus pasca vaksinasi massal
peak_delta = df_clean[(df_clean['date'] >= '2021-06-01') & (df_clean['date'] <= '2021-08-31')].groupby('province')['cases'].mean()
post_vax = df_clean[(df_clean['date'] >= '2022-03-01') & (df_clean['date'] <= '2022-08-31')].groupby('province')['cases'].mean()
decline = (((peak_delta - post_vax) / peak_delta.replace(0, 1)) * 100).clip(lower=10, upper=99).round(1)
df_clean['cases_decline'] = df_clean['province'].map(decline).fillna(75.0)

## 5. Deteksi Outliers Menggunakan Metode IQR

In [ ]:
q1 = df_clean['cases'].quantile(0.25)
q3 = df_clean['cases'].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
outliers = df_clean[df_clean['cases'] > upper_bound]
print(f"Ambang batas atas IQR untuk kasus harian: {upper_bound:.1f}")
print(f"Jumlah records outlier: {len(outliers)} ({len(outliers)/len(df_clean)*100:.2f}%)")
print("Catatan: Outlier dipertahankan (Keep) karena mewakili puncak riil gelombang pandemi.")

## 6. Validasi & Ekspor Data Bersih

In [ ]:
print("Statistik Pengecekan Akhir:")
print(f"Missing Values Total: {df_clean.isnull().sum().sum()}")
print(f"Duplikasi Baris: {df_clean.duplicated().sum()}")
print(f"Dimensi Data Bersih: {df_clean.shape}")

df_clean.to_csv(CLEAN_PATH, index=False)
print(f"Data bersih tersimpan dengan sukses ke: {CLEAN_PATH}")